# 3b. Contract negotiation through the BFF routes

The admin GUI negotiates through `/dsp/current/negotiations/bff-rpc/`: the same eunomia bodies
and answers as the RPCs of `contract_negotiation.ipynb`, each sending one message. The BFF covers
the steps a user decides on (open, accept, agree, terminate); the Consumer's verification and the
Provider's finalization go through their own RPCs (`/dsp/current/negotiations/rpc/`).

This notebook shows the two shortest ways to an agreement and inspects the resulting records
through the Negotiation Agent Management API (`/api/v1/negotiation-agent/*`):

| Part | Who opens | Messages | |
|---|---|---|---|
| I | Consumer | request -> agreement -> verification -> finalization | The Provider agrees on the request as is |
| II | Provider | offer -> acceptance -> agreement -> verification -> finalization | The Consumer accepts the Provider's offer |

**Runs on its own.** It leaves `agreement_id` (from Part II) and records
`agreement_source = ("bff", target_id)`.

## 1. Prerequisites

The guard populates a catalog only when no catalog notebook has run in this kernel; it runs
`setup_catalog_policies.ipynb`, which in turn onboards the participants if needed. Run
`setup_catalog_policies_oauth.ipynb` first to negotiate on its dataset instead.

In [ ]:
if not globals().get("catalog_setup"):
    %run ./setup_catalog_policies.ipynb

### How the negotiation BFF/RPCs and Management API work

Each participant asks **its own agent** to send the next DSP message through an RPC under
`/dsp/current/negotiations/bff-rpc/` or `/dsp/current/negotiations/rpc/`. Every call answers an
`RpcNegotiationResult` with:
- `request`: the RPC body as received.
- `response`: the peer's answer (`ContractNegotiation` ACK for an opening message).
- `negotiationAgentModel`: the caller's own `NegotiationProcessView`, with `id`, `state`, `role`,
  and `correlation` (`consumerPid`, `providerPid`, `callbackAddress`, `peerParticipantId`,
  `identifiers`).

The persisted process, messages, offers and agreement are inspected through the Management API at
`/api/v1/negotiation-agent/{negotiation-processes,negotiation-messages,offers,agreements}`.

In [ ]:
def negotiation_step(response, expected_state):
    """Prints an RpcNegotiationResult and checks the caller's process reached expected_state."""
    answer = show(response)
    process = answer["negotiationAgentModel"]
    assert process["state"] == expected_state, f"expected {expected_state}, got {process['state']}"
    print(f"\n==> {process['state']}")
    return process

# Part I: the Consumer requests, the Provider agrees

## 2. Contract request (`bff-rpc/setup-request-init`, Consumer -> Provider)

The Consumer opens the negotiation on the published offer. DSP 8.2.2.

In [ ]:
payload = {
    "@context": eunomia_context,
    "@type": "RpcNegotiationRequestInitMessage",
    "associatedAgentPeer": data_space_provider_participant_id,
    "providerAddress": data_space_provider_dsp_endpoint,
    "callbackAddress": data_space_consumer_dsp_endpoint,
    "offer": {
        "@id": policy_id,
        "@type": "Offer",
        "target": target_id,
        "permission": [{"action": "use"}],
    },
}
response = requests.post(f"{data_space_consumer}/dsp/current/negotiations/bff-rpc/setup-request-init", headers=json_header, json=payload)
consumer_process = negotiation_step(response, "REQUESTED")
cn_consumer_id = consumer_process["correlation"]["consumerPid"]
cn_provider_id = consumer_process["correlation"]["providerPid"]

def own_negotiation_id(agent_url, consumer_pid):
    """The id an agent gives the negotiation whose consumerPid is `consumer_pid`."""
    processes = requests.get(f"{agent_url}/api/v1/negotiation-agent/negotiation-processes", headers=json_header).json()
    return next(p["id"] for p in processes["items"] if p["correlation"]["consumerPid"] == consumer_pid)

# RPC calls name the negotiation by each agent's own process id, never by DSP pids.
cn_consumer_process_id = consumer_process["id"]
cn_provider_process_id = own_negotiation_id(data_space_provider, cn_consumer_id)
print(f"consumer process: {cn_consumer_process_id}\nprovider process: {cn_provider_process_id}")

## 3. Agreement (`bff-rpc/setup-agreement`, Provider -> Consumer)

The Provider agrees on the requested offer directly, without a counter-offer. DSP 8.3.5.

In [ ]:
payload = {
    "@context": eunomia_context,
    "@type": "RpcNegotiationAgreementMessage",
    "@id": cn_provider_process_id,
}
response = requests.post(f"{data_space_provider}/dsp/current/negotiations/bff-rpc/setup-agreement", headers=json_header, json=payload)
provider_process = negotiation_step(response, "AGREED")

## 4. Verification (`rpc/setup-verification`, Consumer -> Provider)

DSP 8.2.5.

In [ ]:
payload = {
    "@context": eunomia_context,
    "@type": "RpcNegotiationVerificationMessage",
    "@id": cn_consumer_process_id,
}
response = requests.post(f"{data_space_consumer}/dsp/current/negotiations/rpc/setup-verification", headers=json_header, json=payload)
consumer_process = negotiation_step(response, "VERIFIED")

## 5. Finalization (`rpc/setup-finalization`, Provider -> Consumer)

DSP 8.3.6.

In [ ]:
payload = {
    "@context": eunomia_context,
    "@type": "RpcNegotiationFinalizationMessage",
    "@id": cn_provider_process_id,
}
response = requests.post(f"{data_space_provider}/dsp/current/negotiations/rpc/setup-finalization", headers=json_header, json=payload)
provider_process = negotiation_step(response, "FINALIZED")

## 6. Agreement and final state on both sides (`/api/v1/negotiation-agent/*`)

The agreement is retrieved via `GET /api/v1/negotiation-agent/agreements/process/{process_id}` and
`GET /api/v1/negotiation-agent/negotiation-processes/{id}` shows each agent's own view of the
process.

In [ ]:
part1_agreements = show(requests.get(
    f"{data_space_provider}/api/v1/negotiation-agent/agreements/process/{provider_process['id']}",
    headers=json_header,
))
agreement = part1_agreements["items"][0]
agreement_id = agreement["id"]

for name, agent_url, process in [("Consumer", data_space_consumer, consumer_process), ("Provider", data_space_provider, provider_process)]:
    proc_view = requests.get(f"{agent_url}/api/v1/negotiation-agent/negotiation-processes/{process['id']}", headers=json_header).json()
    agr_view = requests.get(f"{agent_url}/api/v1/negotiation-agent/agreements/{agreement_id}", headers=json_header).json()
    print(f"{name}: {proc_view['state']}, agreement {agr_view['id']} ({agr_view['state']})")

# Part II: the Provider offers, the Consumer accepts

## 7. Offer (`bff-rpc/setup-offer-init`, Provider -> Consumer)

The Provider opens the negotiation by sending an offer to the Consumer. DSP 8.3.3. The roles of
the addresses swap with respect to Part I: `associatedAgentPeer` is the Consumer,
`consumerAddress` its DSP base URL, and `callbackAddress` the Provider's own.

In [ ]:
payload = {
    "@context": eunomia_context,
    "@type": "RpcNegotiationOfferInitMessage",
    "associatedAgentPeer": data_space_consumer_participant_id,
    "consumerAddress": data_space_consumer_dsp_endpoint,
    "callbackAddress": data_space_provider_dsp_endpoint,
    "offer": {
        "@id": policy_id,
        "@type": "Offer",
        "target": target_id,
        "permission": [{"action": "use"}],
    },
}
response = requests.post(f"{data_space_provider}/dsp/current/negotiations/bff-rpc/setup-offer-init", headers=json_header, json=payload)
provider_process = negotiation_step(response, "OFFERED")
cn_consumer_id = provider_process["correlation"]["consumerPid"]
cn_provider_id = provider_process["correlation"]["providerPid"]
cn_provider_process_id = provider_process["id"]
cn_consumer_process_id = own_negotiation_id(data_space_consumer, cn_consumer_id)
print(f"consumer process: {cn_consumer_process_id}\nprovider process: {cn_provider_process_id}")

## 8. Acceptance (`bff-rpc/setup-acceptance`, Consumer -> Provider)

DSP 8.2.4.

In [ ]:
payload = {
    "@context": eunomia_context,
    "@type": "RpcNegotiationAcceptanceMessage",
    "@id": cn_consumer_process_id,
}
response = requests.post(f"{data_space_consumer}/dsp/current/negotiations/bff-rpc/setup-acceptance", headers=json_header, json=payload)
consumer_process = negotiation_step(response, "ACCEPTED")

## 9. Agreement (`bff-rpc/setup-agreement`, Provider -> Consumer)

DSP 8.3.5.

In [ ]:
payload = {
    "@context": eunomia_context,
    "@type": "RpcNegotiationAgreementMessage",
    "@id": cn_provider_process_id,
}
response = requests.post(f"{data_space_provider}/dsp/current/negotiations/bff-rpc/setup-agreement", headers=json_header, json=payload)
provider_process = negotiation_step(response, "AGREED")

## 10. Verification and finalization (`rpc/`)

The Consumer verifies (DSP 8.2.5) and the Provider finalizes (DSP 8.3.6).

In [ ]:
payload = {
    "@context": eunomia_context,
    "@type": "RpcNegotiationVerificationMessage",
    "@id": cn_consumer_process_id,
}
response = requests.post(f"{data_space_consumer}/dsp/current/negotiations/rpc/setup-verification", headers=json_header, json=payload)
consumer_process = negotiation_step(response, "VERIFIED")

In [ ]:
payload = {
    "@context": eunomia_context,
    "@type": "RpcNegotiationFinalizationMessage",
    "@id": cn_provider_process_id,
}
response = requests.post(f"{data_space_provider}/dsp/current/negotiations/rpc/setup-finalization", headers=json_header, json=payload)
provider_process = negotiation_step(response, "FINALIZED")

## 11. Agreement and final state on both sides (`/api/v1/negotiation-agent/*`)

The agreement is the last offer both sides accepted, now with `assigner` (Provider) and `assignee`
(Consumer). Both agents keep the same agreement id; it is the `agreementId` a transfer request
names.

In [ ]:
part2_agreements = show(requests.get(
    f"{data_space_provider}/api/v1/negotiation-agent/agreements/process/{provider_process['id']}",
    headers=json_header,
))
agreement = part2_agreements["items"][0]
agreement_id = agreement["id"]

for name, agent_url, process in [("Consumer", data_space_consumer, consumer_process), ("Provider", data_space_provider, provider_process)]:
    proc_view = requests.get(f"{agent_url}/api/v1/negotiation-agent/negotiation-processes/{process['id']}", headers=json_header).json()
    msgs_view = requests.get(f"{agent_url}/api/v1/negotiation-agent/negotiation-messages/process/{process['id']}", headers=json_header).json()
    agr_view = requests.get(f"{agent_url}/api/v1/negotiation-agent/agreements/{agreement_id}", headers=json_header).json()
    print(f"{name}: {proc_view['state']}, messages={len(msgs_view['items'])}, agreement {agr_view['id']} ({agr_view['state']})")

In [ ]:
# Marker read by the guard of bff_push_flow.ipynb
agreement_source = ("bff", target_id)
print(f"agreement_id = {agreement_id}")